<a href="https://colab.research.google.com/github/andiandiandika3-prog/Mechine-Learning/blob/main/1_Random%20Forest/Model_train_test_hyperparamtertunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Secara umum Random Forest merupakan pengembangan dari Bagging, yaitu secara berkali-kali melakukan bootstrap terhadap data training dan menyusun pohon klasifikasi berdasarkan data hasil resampling tersebut, dan kemudian proses prediksi dilakukan dengan mengagregasi hasil prediksi dari banyak pohon yang umumnya menggunakan pendekatan majority vote.

# Algoritma dasar dari random forest adalah sebagai berikut:
1.  Given a training data set
2.  Select number of trees to build (n_trees)
3.  for i = 1 to n_trees do
4.  |  Generate a bootstrap sample of the original data
5.  |  Grow a classification tree to the bootstrapped data
6.  |  for each split do
7.  |  | Select m_try variables at random from all p variables
8.  |  | Pick the best variable/split-point among the m_try
9.  |  | Split the node into two child nodes
10. |  end
11. | Use typical tree model stopping criteria to determine when a
    | tree is complete (but do not prune)
12. end
13. Output ensemble of trees

# Template untuk download data dari gdrive (csv)

https://drive.google.com/uc?export=download&id=1X_G95k-GmbDhgC1bH92Oj8-STatG8iDy


# Muat data

In [43]:
df <- read.csv("https://drive.google.com/uc?export=download&id=1X_G95k-GmbDhgC1bH92Oj8-STatG8iDy")

cat("-----------------------------------------","\n")
cat("cek struktur data","\n")
cat("-----------------------------------------","\n")
str(df)

cat("-----------------------------------------","\n")
cat("Tampilkan 5 Data Awal","\n")
cat("-----------------------------------------","\n")
head(df)


cat("----------------------------------------------------","\n")
cat("Membuat variabel target quality [0 <= 6, 1 sisanya] ","\n")
cat("----------------------------------------------------","\n")
df$target <- ifelse(df$quality <= 6,0,1)
head(df)

----------------------------------------- 
cek struktur data 
----------------------------------------- 
'data.frame':	1599 obs. of  12 variables:
 $ fixed.acidity       : num  7.4 7.8 7.8 11.2 7.4 7.4 7.9 7.3 7.8 7.5 ...
 $ volatile.acidity    : num  0.7 0.88 0.76 0.28 0.7 0.66 0.6 0.65 0.58 0.5 ...
 $ citric.acid         : num  0 0 0.04 0.56 0 0 0.06 0 0.02 0.36 ...
 $ residual.sugar      : num  1.9 2.6 2.3 1.9 1.9 1.8 1.6 1.2 2 6.1 ...
 $ chlorides           : num  0.076 0.098 0.092 0.075 0.076 0.075 0.069 0.065 0.073 0.071 ...
 $ free.sulfur.dioxide : num  11 25 15 17 11 13 15 15 9 17 ...
 $ total.sulfur.dioxide: num  34 67 54 60 34 40 59 21 18 102 ...
 $ density             : num  0.998 0.997 0.997 0.998 0.998 ...
 $ pH                  : num  3.51 3.2 3.26 3.16 3.51 3.51 3.3 3.39 3.36 3.35 ...
 $ sulphates           : num  0.56 0.68 0.65 0.58 0.56 0.56 0.46 0.47 0.57 0.8 ...
 $ alcohol             : num  9.4 9.8 9.8 9.8 9.4 9.4 9.4 10 9.5 10.5 ...
 $ quality             : int  5 

,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,alcohol,quality
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5
2,7.8,0.88,0.00,2.6,0.098,25,67,0.9968,3.20,0.68,9.8,5
3,7.8,0.76,0.04,2.3,0.092,15,54,0.9970,3.26,0.65,9.8,5
4,11.2,0.28,0.56,1.9,0.075,17,60,0.9980,3.16,0.58,9.8,6
5,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5
6,7.4,0.66,0.00,1.8,0.075,13,40,0.9978,3.51,0.56,9.4,5


---------------------------------------------------- 
Membuat variabel target quality [0 <= 6, 1 sisanya]  
---------------------------------------------------- 


,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,alcohol,quality,target
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5,0
2,7.8,0.88,0.00,2.6,0.098,25,67,0.9968,3.20,0.68,9.8,5,0
3,7.8,0.76,0.04,2.3,0.092,15,54,0.9970,3.26,0.65,9.8,5,0
4,11.2,0.28,0.56,1.9,0.075,17,60,0.9980,3.16,0.58,9.8,6,0
5,7.4,0.70,0.00,1.9,0.076,11,34,0.9978,3.51,0.56,9.4,5,0
6,7.4,0.66,0.00,1.8,0.075,13,40,0.9978,3.51,0.56,9.4,5,0


# Model umum

In [44]:
# install pakege
if(!require(randomForest)){
  install.packages("randomForest")
  library(randomForest)
}

randomForest(as.factor(target)~.,
                       data = df,
                       ntree =10, # Jumlah pohon
                       mtry = 3) # jumlah variabel kandidat yang boleh dilihat pada setiap split node


Call:
 randomForest(formula = as.factor(target) ~ ., data = df, ntree = 10,      mtry = 3) 
               Type of random forest: classification
                     Number of trees: 10
No. of variables tried at each split: 3

        OOB estimate of  error rate: 0.89%
Confusion matrix:
     0   1 class.error
0 1362   4 0.002928258
1   10 204 0.046728972

# Model dengan data train dan test

## Membuat data train dan testing

In [45]:
set.seed(123) # Agar hasil random selalu sama

train_index <- sample(1:nrow(df),0.8*nrow(df)) # ambil sampel 80% secara acak

x_train <- df[train_index,!(names(df) %in% c("target","quality"))]
y_train <- df[train_index,(names(df) %in% "target")]

x_test <- df[-train_index,!(names(df) %in% c("target","quality"))]
y_test <- df[-train_index,(names(df) %in% "target")]

cat("----------------------------------------------------","\n")
cat("Panjang Data train dan test ","\n")
cat("----------------------------------------------------","\n")
cat("Panjang data     : ", nrow(df),"\n")
cat("Panjang train    : ", nrow(x_train),"\n")
cat("Panjang test     : ", nrow(x_test),"\n")

---------------------------------------------------- 
Panjang Data train dan test  
---------------------------------------------------- 
Panjang data     :  1599 
Panjang train    :  1279 
Panjang test     :  320 


## Membuat Model dengan Data Training

In [46]:
# Bangun model
model_rf <-randomForest(
  x = x_train,
  y = as.factor(y_train),
  ntree = 1000,
  mtry = 3)

## Buat prediksi

In [47]:
hasil <- predict(model_rf, newdata = x_test)

table(prediksi = hasil,
      aktual = y_test)

        aktual
prediksi   0   1
       0 285  14
       1   6  15

# Hyperparameter Tunning

Grid Search (sederhana), Bayesian (via rBayesianOptimization), Optuna (via reticulate), dan metode lainnya seperti Random Search.

| Metode        | Cara Pencarian  | Strategi Pemilihan Kombinasi                                              | Contoh untuk RF Kamu                                                 |
| ------------- | --------------- | ------------------------------------------------------------------------- | -------------------------------------------------------------------- |
| Grid Search   | Eksaustif       | Coba semua kombinasi dari grid yang didefinisikan                         | mtry=c(2,3,4), ntree=c(100,500) → 12 kombinasi pasti itbox           |
| Random Search | Random Sampling | Ambil N sampel acak dari ruang parameter                                  | mtry uniform(1,6), ntree uniform(100,1000) → 20 sampel acak kirim+1  |
| Bayesian      | Probabilistik   | Bangun model dari hasil sebelumnya, prediksi mana yang paling menjanjikan | Trial 1-5 random, lalu fokus ke daerah bagus (mtry~3)                |

| Aspek        | Grid          | Random             | Bayesian    |
| ------------ | ------------- | ------------------ | ----------- |
| Evaluasi     | Selalu sama   | Fixed N sampel     | Adaptif     |
| Waktu        | Lambat        | Cepat              | Sedang      |
| Kualitas     | Baseline      | > Grid untuk dim>3 | Sangat baik |
| Memori       | Tidak belajar | Tidak              | Belajar     |
| Dataset Kamu | OK (12 eval)  | Recommended awal   | Terbaik     |

## Grid Search (caret)

In [48]:
if(!(require(caret))){
  install.packages("caret")
  library(caret)
}

if(!(require(ranger))){
  install.packages("ranger")
  library(ranger)
}

set.seed(123)

ctrl <- trainControl(method = "cv",number = 5) # menggunakan k-fold cross validation dengan nilai k = 5

grid_param <- expand.grid(
  mtry = c(2:6),
  splitrule = "gini",
  # gini(Gini Impurity (default) = Classification - pisah kelas sehomogen mungkin
  # extratrees (Extreme Randomized)= Lebih cepat, sedikit random
  # "hellinger" (Hellinger Distance) = Imbalanced data
  min.node.size = 10
)


tune_grid <- train(
  x = x_train,
  y = as.factor(y_train),
  method = "ranger", #Random Forest (lebih cepat dari randomForest)
  trControl = ctrl,
  tuneGrid = grid_param,
  num.trees = 1000
)

cat("----------------------------------------------------","\n")
cat("Kombinasi Parameter Terbaik Grid Search","\n")
cat("----------------------------------------------------","\n")
print(tune_grid$bestTune)

cat("----------------------------------------------------","\n")
cat("Prediksi menggunakan Grid Search","\n")
cat("----------------------------------------------------","\n")

pred_test <- predict(tune_grid, newdata = x_test) # Changed new_data to newdata

confusionMatrix(
  pred_test,
  as.factor(y_test)
)

---------------------------------------------------- 
Kombinasi Parameter Terbaik Grid Search 
---------------------------------------------------- 
  mtry splitrule min.node.size
3    4      gini            10
---------------------------------------------------- 
Prediksi menggunakan Grid Search 
---------------------------------------------------- 


Confusion Matrix and Statistics

          Reference
Prediction   0   1
         0 285  13
         1   6  16
                                          
               Accuracy : 0.9406          
                 95% CI : (0.9088, 0.9639)
    No Information Rate : 0.9094          
    P-Value [Acc > NIR] : 0.02694         
                                          
                  Kappa : 0.5959          
                                          
 Mcnemar's Test P-Value : 0.16867         
                                          
            Sensitivity : 0.9794          
            Specificity : 0.5517          
         Pos Pred Value : 0.9564          
         Neg Pred Value : 0.7273          
             Prevalence : 0.9094          
         Detection Rate : 0.8906          
   Detection Prevalence : 0.9313          
      Balanced Accuracy : 0.7656          
                                          
       'Positive' Class : 0               
                              

# Random Search (caret)
Lebih cepat dari grid, sampling acak dari ruang parameter.



In [49]:
tune_random <- train(
  y = as.factor(y_train),
  x = x_train,
  method = "ranger",
  trControl = ctrl,
  tuneLength = 20, # Coba 20 kombinasi acak
  num.trees = 500
)

cat("----------------------------------------------------","\n")
cat("Kombinasi Parameter Terbaik Random Search","\n")
cat("----------------------------------------------------","\n")
print(tune_random$bestTune)

cat("----------------------------------------------------","\n")
cat("Prediksi menggunakan Random Search","\n")
cat("----------------------------------------------------","\n")
pred_test <- predict(tune_random, newdata = x_test) # Changed new_data to newdata

confusionMatrix(
  pred_test,
  as.factor(y_test)
)

note: only 10 unique complexity parameters in default grid. Truncating the grid to 10 .

---------------------------------------------------- 
Kombinasi Parameter Terbaik Random Search 
---------------------------------------------------- 
   mtry  splitrule min.node.size
10    6 extratrees             1
---------------------------------------------------- 
Prediksi menggunakan Random Search 
---------------------------------------------------- 


Confusion Matrix and Statistics

          Reference
Prediction   0   1
         0 285  14
         1   6  15
                                          
               Accuracy : 0.9375          
                 95% CI : (0.9051, 0.9614)
    No Information Rate : 0.9094          
    P-Value [Acc > NIR] : 0.04359         
                                          
                  Kappa : 0.567           
                                          
 Mcnemar's Test P-Value : 0.11752         
                                          
            Sensitivity : 0.9794          
            Specificity : 0.5172          
         Pos Pred Value : 0.9532          
         Neg Pred Value : 0.7143          
             Prevalence : 0.9094          
         Detection Rate : 0.8906          
   Detection Prevalence : 0.9344          
      Balanced Accuracy : 0.7483          
                                          
       'Positive' Class : 0               
                              

# Bayesian Optimization (rBayesianOptimization)
Cerdas, gunakan model probabilistik untuk prediksi kombinasi terbaik selanjutnya.


OOB Error = Out-of-Bag = CV bawaan Random Forest (setiap pohon test sampel yang tidak masuk bootstrap-nya).

| Acq | Rumus            | Eksplorasi  | Eksploitasi | Kompleksitas | Stability |
| --- | ---------------- | ----------- | ----------- | ------------ | --------- |
| UCB | μ + κσ           | Baik        | Baik        | Simple       | ⭐⭐⭐⭐⭐     |
| EI  | E[improvement]   | Sangat Baik | Sangat Baik | Kompleks     | ⭐⭐⭐⭐      |
| POI | P(improvement>0) | Kurang      | Baik        | Simple       | ⭐⭐⭐       |

In [50]:
if(!require(rBayesianOptimization)){
  install.packages("rBayesianOptimization")
  library(rBayesianOptimization)
}

# Fungsi evaluasi untuk bayesian optimization
rf_bayes <- function(mtry, min.node.size, sample.fraction){
 model <- ranger(
  x = x_train,
  y = as.factor(y_train),

  mtry = round(mtry),
  min.node.size = round(min.node.size),
  sample.fraction = sample.fraction, # proporsi boostrap sample

  num.trees = 500
   )

   pred <- predict(model, data = x_test)$predictions

   acc <- confusionMatrix(
    as.factor(pred),
    as.factor(y_test)
   )$overall["Accuracy"]

   list(
    Score = acc,
    Pred = pred
   )
}

# Bayesian Optimization
bayes_tunning <- BayesianOptimization(
  FUN = rf_bayes,

  bounds = list(
    mtry = c(2L, ncol(x_train)),
    min.node.size = c(1L, 10L),#L = integer
    sample.fraction = c(0.5, 1)
  ),

  init_points = 10,
  n_iter = 20,

  acq = "ucb",
  kappa = 2.576,

  verbose = TRUE
)

cat("----------------------------------------------------","\n")
cat("Kombinasi Parameter Terbaik Bayesian Optimization","\n")
cat("----------------------------------------------------","\n")

print(bayes_tunning$Best_Par)

cat("----------------------------------------------------","\n")
cat("Training model dengan bayesian","\n")
cat("----------------------------------------------------","\n")

best_param <- bayes_tunning$Best_Par

rf_bayes_final <- ranger(
  x = x_train,
  y = as.factor(y_train),

  mtry = round(best_param["mtry"]),
  min.node.size = round(best_param["min.node.size"]),
  sample.fraction = best_param["sample.fraction"],

  num.trees = 500

)

cat("----------------------------------------------------","\n")
cat("Prediksi menggunakan bayesian","\n")
cat("----------------------------------------------------","\n")

pred_test <- predict(rf_bayes_final, data = x_test)$predictions

confusionMatrix(
  as.factor(pred_test),
  as.factor(y_test)
)


elapsed = 0.308	Round = 1	mtry = 4.0000	min.node.size = 8.0000	sample.fraction = 0.8608845	Value = 0.928125 
elapsed = 0.446	Round = 2	mtry = 3.0000	min.node.size = 9.0000	sample.fraction = 0.907394	Value = 0.93125 
elapsed = 0.737	Round = 3	mtry = 7.0000	min.node.size = 5.0000	sample.fraction = 0.9275184	Value = 0.940625 
elapsed = 0.752	Round = 4	mtry = 11.0000	min.node.size = 1.0000	sample.fraction = 0.6489327	Value = 0.940625 
elapsed = 0.401	Round = 5	mtry = 8.0000	min.node.size = 3.0000	sample.fraction = 0.6870018	Value = 0.940625 
elapsed = 0.309	Round = 6	mtry = 7.0000	min.node.size = 4.0000	sample.fraction = 0.5397275	Value = 0.928125 
elapsed = 0.342	Round = 7	mtry = 7.0000	min.node.size = 9.0000	sample.fraction = 0.6434905	Value = 0.94375 
elapsed = 0.284	Round = 8	mtry = 5.0000	min.node.size = 9.0000	sample.fraction = 0.67071	Value = 0.921875 
elapsed = 0.367	Round = 9	mtry = 9.0000	min.node.size = 9.0000	sample.fraction = 0.5859684	Value = 0.9375 
elapsed = 0.554	Round = 1

Confusion Matrix and Statistics

          Reference
Prediction   0   1
         0 284  11
         1   7  18
                                          
               Accuracy : 0.9438          
                 95% CI : (0.9126, 0.9663)
    No Information Rate : 0.9094          
    P-Value [Acc > NIR] : 0.01584         
                                          
                  Kappa : 0.6361          
                                          
 Mcnemar's Test P-Value : 0.47950         
                                          
            Sensitivity : 0.9759          
            Specificity : 0.6207          
         Pos Pred Value : 0.9627          
         Neg Pred Value : 0.7200          
             Prevalence : 0.9094          
         Detection Rate : 0.8875          
   Detection Prevalence : 0.9219          
      Balanced Accuracy : 0.7983          
                                          
       'Positive' Class : 0               
                              